In [ ]:
import matplotlib.pyplot as plt
print(plt.style.available)

In [ ]:
import matplotlib.pyplot as plt
plt.style.use('ggplot')  # Or choose another style from plt.style.available

In [ ]:
import seaborn as sns
sns.set_style('darkgrid')  # or any other seaborn style

In [ ]:
import numpy as np
import pandas as pd
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.dates import DateFormatter
import datetime as dt
from statsmodels.tsa.stattools import adfuller
import plotly.offline as pyoff
import plotly.graph_objs as go

# Check if 'seaborn-darkgrid' is available, otherwise use 'ggplot'
import matplotlib
available_styles = plt.style.available
if 'seaborn-darkgrid' in available_styles:
    matplotlib.style.use('seaborn-darkgrid')
else:
    matplotlib.style.use('ggplot')

%matplotlib inline
import warnings
warnings.filterwarnings("ignore")


#Importing complete wind and solar energy  data for 2015- 2021 Germany

In [ ]:
df = pd.read_csv("time_series_60min_singleindex_filtered.csv", parse_dates=[0], index_col=0)

In [ ]:
df.info()

In [ ]:
#display rows which have null values in "wind generation actual"
df[df['DE_wind_generation_actual'].isna()]

In [ ]:
df[df['DE_solar_generation_actual'].isna()]  #display rows which have null values in "solar generation actual"

In [ ]:
df.isna().sum()   #75 null values for wind_gen_actual #104 null values for solar_gen_actual 

#Filling null values from the values of the day before 


In [ ]:
nulls = df.isna().any(axis=1)
df.loc[nulls, ['DE_solar_generation_actual','cet_cest_timestamp']] = df.shift(24).loc[nulls, ['DE_solar_generation_actual','cet_cest_timestamp']]

#shift function to shift the index 24 hours

#Rechecking the null values (whatever left)

In [ ]:
df[df['DE_solar_generation_actual'].isna()]

Observation --> The only remaining null values left are the ones that were from the first day because we filled in the rest of the null values from the day before.

In [ ]:
#fill in the rest to zero because that is what they would be at those hours of night till 6 am
df['DE_solar_generation_actual'].fillna(0, inplace = True)
df['cet_cest_timestamp'].fillna(0, inplace = True) 

####Repeating the same for wind energy

In [ ]:
nulls =df.isna().any(axis=1)
df.loc[nulls, ['DE_wind_generation_actual','cet_cest_timestamp']] = df.shift(24).loc[nulls, ['DE_wind_generation_actual','cet_cest_timestamp']]



####Replacing the null values (whatever left) by mean



In [ ]:
df[df['DE_wind_generation_actual'].isna()]

In [ ]:
df['DE_wind_generation_actual'].mean()

In [ ]:
df['DE_wind_generation_actual'].fillna(11556, inplace=True)

In [ ]:
df.isna().sum()

#HEAT MAP TO CHECK CORRELATIONS BETWEEN FEATURES

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Select only numeric columns
numeric_df = df.select_dtypes(include=[float, int])

# Create the heatmap
fig, ax = plt.subplots(1, 1, figsize=(12, 12))
sns.heatmap(data=numeric_df.corr(),
            annot=True,
            annot_kws={"size": 10},
            square=True,
            cmap='seismic',
            fmt='.2f',
            linewidths=0.5,
            linecolor='w',
            ax=ax)
plt.title('Correlation Analysis')
plt.show()


solar_generation_actual depends most on solar_profile

wind_generation-actual depends most on wind_profile, wind_onshore_profile, wind_onshore_generation and  then on wind_offshore_profile and wind_offshore_generation 

#dataset with timestamps as features (used later for time series forecasting models)

In [ ]:
modified=df[['cet_cest_timestamp', 'DE_solar_generation_actual','DE_wind_generation_actual']]
modified

In [ ]:
#modified dataset
# energy_all = pd.read_csv("time_series_60min_singleindex_filtered (3).csv",
#                         parse_dates=[0], index_col=0)

The only remaining null values left are the ones that were from the first day because we filled in the rest of the null values from the day before. We see that it that it also turned the dummy column, cet_cest_timestamp, into nulls at the same location. This is why I used it as dummy column.

In [ ]:
# Drop the column becasue we do not need another time column
modified.drop(columns='cet_cest_timestamp',inplace=True)

# EDA

##Histograms for solar_generation_actual and wind_generation_actual 

In [ ]:
plt.figure(figsize=(12,5))
sns.histplot(df[df['DE_solar_generation_actual'] != 0].DE_solar_generation_actual)  # remove all 0 in histogram
plt.title('Histogram of power')
plt.xlabel('Power ($W/m^2$)')
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
sns.histplot(df[df['DE_wind_generation_actual'] != 0].DE_wind_generation_actual)  # remove all 0 in histogram
plt.title('Histogram of power')
plt.xlabel('Power ($W/m^2$)')
plt.show()

In [ ]:
# We will create a new data frame so that we can make the appropriate boxplots. 
DE_energy = modified.reset_index()
DE_energy.info()

In [ ]:
# create utc_timestamp as a column and another hour column
DE_energy['utc_timestamp'] = pd.to_datetime(DE_energy['utc_timestamp']).apply(lambda x: dt.datetime.strftime(x,'%Y-%m-%d %H:%M:%S'))

DE_energy['utc_timestamp']=pd.to_datetime(DE_energy['utc_timestamp'])
DE_energy['hour'] = DE_energy['utc_timestamp'].dt.hour

## boxplot of energy output vs the hour at which recorded (useful for time series forecast)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Boxplot of the average solar generation production
plt.figure(figsize=(10, 5))
sns.boxplot(x='hour', y='DE_solar_generation_actual', data=DE_energy)

plt.title('Average Daily Solar Production', fontsize=20)
plt.xlabel('Time', fontsize=20)
plt.ylabel("MWh", rotation=0, ha='right', fontsize=20)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Boxplot of the average wind generation production
plt.figure(figsize=(10, 5))
sns.boxplot(x='hour', y='DE_wind_generation_actual', data=DE_energy)

plt.title('Average Daily Wind Production', fontsize=20)
plt.xlabel('Time', fontsize=20)
plt.ylabel("MWh", rotation=0, ha='right', fontsize=20)
plt.show()


Wind appears to not differ as much during the day as solar.The time of day does affect wind speed with surface temperatures increasing, however it is not shown in this plot.

We can use mean over all the days to get a better time series analysis with less noise.

In [ ]:
# use the resample function to average the generation over all the days
modified = modified.resample('D').mean()
modified

In [ ]:
modified.shape

#time series plot for energy averged over all days

In [ ]:
# visualize the time time series of solar 
plt.figure(figsize=(15,5))
plt.plot(modified['DE_solar_generation_actual'])
plt.title('Solar Generation in MW', fontsize=20)
plt.ylabel('MW')
plt.xlabel('date')
ax = plt.gca()

ax.autoscale(enable=True, axis='x', tight=True)
ax.xaxis.set_major_formatter(DateFormatter("%Y %b"))
plt.show()

In [ ]:
# Do the same thing for the wind data
plt.figure(figsize=(15,5))
plt.plot(modified['DE_wind_generation_actual'], c='red')
plt.title('Wind Generation in MW', fontsize=20)
plt.ylabel('MW')
plt.xlabel('date')
ax = plt.gca()

ax.autoscale(enable=True, axis='x', tight=True)
ax.xaxis.set_major_formatter(DateFormatter("%Y %b"))
plt.show()

In [ ]:
# Lets check out some basic statistics for the data
modified.describe()

The mean is much higher for wind and that is because wind produces much more electricity than solar in Germany. They both have very high standard deviations.

#MACHINE LEARNING ALOGORITHMS 

##Train-Test-Split

In [ ]:
df.head()

In [ ]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [ ]:
df.fillna(0, inplace=True)

In [ ]:
from sklearn.model_selection import train_test_split
X1_train, X1_test, y1_train, y1_test = train_test_split(df['DE_solar_profile'], df['DE_solar_generation_actual'],test_size = 0.2, random_state = None)

In [ ]:
X1_train=np.array(X1_train).reshape(-1,1)
X1_test=np.array(X1_test).reshape(-1,1)

In [ ]:
X1_train

In [ ]:
X2_train, X2_test, y2_train, y2_test = train_test_split(df['DE_wind_profile'], df['DE_wind_generation_actual'],test_size = 0.2, random_state = None)

In [ ]:
# For Wind Energy
X2_train=np.array(X2_train).reshape(-1,1)
X2_test=np.array(X2_test).reshape(-1,1)

In [ ]:
X2_train

##Lasso Regression

In [ ]:
df.head()

In [ ]:
from sklearn.linear_model import Lasso
 
# Train the model
# Solar Energy
lasso = Lasso(alpha = 1)
lasso.fit(X1_train, y1_train)
y1_pred = lasso.predict(X1_test)

In [ ]:
from sklearn.linear_model import Lasso
import numpy as np

# Ensure X2_train and X2_test are in the correct shape
X2_train = X2_train.values.reshape(-1, 1) if X2_train.ndim == 1 else X2_train
X2_test = X2_test.values.reshape(-1, 1) if X2_test.ndim == 1 else X2_test

# Wind energy
lasso = Lasso(alpha=1)
lasso.fit(X2_train, y2_train)
y2_pred = lasso.predict(X2_test)


In [ ]:
#Hyperparameter tuning - alpha = 2
lasso = Lasso(alpha = 2)
lasso.fit(X1_train, y1_train)
y1_pred2 = lasso.predict(X1_test)

In [ ]:
lasso = Lasso(alpha = 2)
lasso.fit(X2_train, y2_train)
y2_pred2 = lasso.predict(X2_test)

###Evaluation Metrics

####Mean Squared Error

In [ ]:
#alpha = 1
from sklearn.metrics import mean_squared_error
print("Solar MSE = ",mean_squared_error(y1_test,y1_pred))

In [ ]:
print("Wind MSE = ",mean_squared_error(y2_test,y2_pred))

In [ ]:
#alpha = 2
print("Solar MSE = ",mean_squared_error(y1_test,y1_pred2))

In [ ]:
print("Wind MSE = ",mean_squared_error(y2_test,y2_pred2))

####Mean Absolute Error

In [ ]:
from sklearn.metrics import mean_absolute_error
print("Solar MAE = ",mean_absolute_error(y1_test,y1_pred))

In [ ]:
print("Wind MAE = ",mean_absolute_error(y2_test,y2_pred))

In [ ]:
#alpha = 2
print("Solar MAE = ",mean_absolute_error(y1_test,y1_pred2))

In [ ]:
print("Wind MAE = ",mean_absolute_error(y2_test,y2_pred2))

####Root Mean Squared Error

In [ ]:
print("Solar RMSE = ",np.sqrt(mean_squared_error(y1_test,y1_pred)))

In [ ]:
print("Wind RMSE = ",np.sqrt(mean_squared_error(y2_test,y2_pred)))

In [ ]:
#alpha = 2
print("Solar RMSE = ",np.sqrt(mean_squared_error(y1_test,y1_pred2)))

In [ ]:
print("Wind RMSE = ",np.sqrt(mean_squared_error(y2_test,y2_pred)))

####R Squared

In [ ]:
from sklearn.metrics import r2_score
r2_solar = r2_score(y1_test,y1_pred)
print("Solar R2 = ",r2_solar)

In [ ]:
r2_wind = r2_score(y2_test,y2_pred)
print("Wind R2 = ",r2_wind)

In [ ]:
#alpha = 2
r2_solar = r2_score(y1_test,y1_pred2)
print("Solar R2 = ",r2_solar)

In [ ]:
r2_wind = r2_score(y2_test,y2_pred2)
print("Wind R2 = ",r2_wind)

###Plot

In [ ]:
g=plt.plot(y1_test - y1_pred,marker='o',linestyle='')

#RIDGE REGRESSION

In [ ]:
from sklearn.linear_model import Ridge
 
# Train the model
ridgeSolar = Ridge(alpha = 1)
ridgeSolar.fit(X1_train, y1_train)
y1_pred = ridgeSolar.predict(X1_test)

In [ ]:
ridgeWind = Ridge(alpha = 1)
ridgeWind.fit(X2_train, y2_train)
y2_pred = ridgeSolar.predict(X2_test)

In [ ]:
#Hyperparameter tuning - alpha = 2
ridgeSolar = Ridge(alpha = 2)
ridgeSolar.fit(X1_train, y1_train)
y1_pred2 = ridgeSolar.predict(X1_test)

In [ ]:
ridgeWind = Ridge(alpha = 2)
ridgeWind.fit(X2_train, y2_train)
y2_pred2 = ridgeSolar.predict(X2_test)

##Evaluation Metrics

###Mean Squared Error

In [ ]:
from sklearn.metrics import mean_squared_error
print("Solar MSE = ",mean_squared_error(y1_test,y1_pred))

In [ ]:
print("Wind MSE = ",mean_squared_error(y2_test,y2_pred))

In [ ]:
#alpha = 2
print("Solar MSE = ",mean_squared_error(y1_test,y1_pred2))

In [ ]:
print("Wind MSE = ",mean_squared_error(y2_test,y2_pred2))

###Mean Absolute Error

In [ ]:
from sklearn.metrics import mean_absolute_error
print("Solar MAE = ",mean_absolute_error(y1_test,y1_pred))

In [ ]:
print("Wind MAE = ",mean_absolute_error(y2_test,y2_pred))

In [ ]:
#alpha = 2
print("Solar MAE = ",mean_absolute_error(y1_test,y1_pred2))

In [ ]:
print("Wind MAE = ",mean_absolute_error(y2_test,y2_pred2))

###Root Mean Squared Error

In [ ]:
print("Solar RMSE = ",np.sqrt(mean_squared_error(y1_test,y1_pred)))

In [ ]:
print("Wind RMSE = ",np.sqrt(mean_squared_error(y2_test,y2_pred)))

In [ ]:
#alpha = 2
print("Solar RMSE = ",np.sqrt(mean_squared_error(y1_test,y1_pred2)))

In [ ]:
print("Wind RMSE = ",np.sqrt(mean_squared_error(y2_test,y2_pred2)))

###R Squared

In [ ]:
from sklearn.metrics import r2_score
r2_solar = r2_score(y1_test,y1_pred)
print("Solar R2 = ",r2_solar)

In [ ]:
r2_wind = r2_score(y2_test,y2_pred)
print("Wind R2 = ",r2_wind)

In [ ]:
#alpha = 2
r2_solar = r2_score(y1_test,y1_pred2)
print("Solar R2 = ",r2_solar)

In [ ]:
r2_wind = r2_score(y2_test,y2_pred2)
print("Wind R2 = ",r2_wind)

###Plot

In [ ]:
g=plt.plot(y1_test - y1_pred,marker='o',linestyle='')

#DECISION TREE

In [ ]:
from sklearn.tree import DecisionTreeRegressor 
  
# create a regressor object
regressor = DecisionTreeRegressor(random_state = 0) 
  
# fit the regressor with X and Y data
regressor.fit(X1_train, y1_train)

In [ ]:
y1_pred = regressor.predict(X1_test)
  
# print the predicted price
y1_test


In [ ]:
y1_pred

In [ ]:
#from sklearn.model_selection import cross_val_score 
#cross_val_score(regressor, X1_train, y1_train, cv=50)

In [ ]:
#for wind energy
regressor.fit(X2_train, y2_train)

In [ ]:
y2_pred = regressor.predict(X2_test)
  
# print the predicted price
y2_test

In [ ]:
y2_pred

In [ ]:
#from sklearn.model_selection import cross_val_score 
#cross_val_score(regressor, X2_train, y2_train, cv=50)

In [ ]:
from sklearn.metrics import r2_score

In [ ]:
r2_score(y1_pred, y1_test)

In [ ]:
r2_score(y2_pred, y2_test)

In [ ]:
from sklearn.metrics import mean_absolute_error as mae

In [ ]:
error = mae(y1_test, y1_pred)
print(error)

In [ ]:
error = mae(y2_test, y2_pred)
print(error)

In [ ]:
from sklearn.metrics import mean_squared_error
  
MSE = mean_squared_error(y1_test, y1_pred)
print(MSE)

In [ ]:
from sklearn.metrics import mean_squared_error
  
MSE = mean_squared_error(y2_test, y2_pred)
print(MSE)

In [ ]:
#root mean squared error for solar
print("RMSE",np.sqrt(mean_squared_error(y1_test,y1_pred)))

In [ ]:
#root mean squared error for wind
print("RMSE",np.sqrt(mean_squared_error(y2_test,y2_pred)))

Hyperparameter tuning with random state=1

In [ ]:
from sklearn.tree import DecisionTreeRegressor 
  
# create a regressor object
regressor = DecisionTreeRegressor(random_state = 1) 
  
# fit the regressor with X and Y data
regressor.fit(X1_train, y1_train)

In [ ]:
y1_pred2 = regressor.predict(X1_test)

In [ ]:
#for wind energy
regressor.fit(X2_train, y2_train)

In [ ]:
y2_pred2 = regressor.predict(X2_test)

In [ ]:
r2_score(y1_pred2, y1_test)

In [ ]:
r2_score(y2_pred2, y2_test)

In [ ]:
MSE = mean_squared_error(y1_test, y1_pred2)
print(MSE)

In [ ]:
MSE = mean_squared_error(y2_test, y2_pred2)
print(MSE)

In [ ]:
#root mean squared error for solar
print("RMSE",np.sqrt(mean_squared_error(y1_test,y1_pred2)))

In [ ]:
#root mean squared error for solar
print("RMSE",np.sqrt(mean_squared_error(y2_test,y2_pred2)))

In [ ]:
error = mae(y1_test, y1_pred2)
print(error)

In [ ]:
error = mae(y2_test, y2_pred2)
print(error)

In [ ]:
g=plt.plot(y1_test - y1_pred,marker='o',linestyle='')

LSTM MODEL

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

In [ ]:
# Function to preprocess the data for RNN
def preprocess_rnn_data(df, feature_column, target_column, time_steps=24):
    data = df[[feature_column, target_column]].values
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(data)
    
    X, y = [], []
    for i in range(len(scaled_data) - time_steps):
        X.append(scaled_data[i:i + time_steps])
        y.append(scaled_data[i + time_steps, 1])
    
    X = np.array(X)
    y = np.array(y)
    
    return X, y, scaler

In [ ]:
# Preprocess data for solar generation
X_solar, y_solar, solar_scaler = preprocess_rnn_data(df, 'DE_solar_profile', 'DE_solar_generation_actual')

In [ ]:
# Preprocess data for wind generation
X_wind, y_wind, wind_scaler = preprocess_rnn_data(df, 'DE_wind_profile', 'DE_wind_generation_actual')

In [ ]:
# Split the data into training and testing sets
X_solar_train, X_solar_test, y_solar_train, y_solar_test = train_test_split(X_solar, y_solar, test_size=0.2, random_state=42)
X_wind_train, X_wind_test, y_wind_train, y_wind_test = train_test_split(X_wind, y_wind, test_size=0.2, random_state=42)

In [ ]:
# Build the RNN model for solar generation
model_solar = Sequential()
model_solar.add(LSTM(units=50, return_sequences=True, input_shape=(X_solar_train.shape[1], X_solar_train.shape[2])))
model_solar.add(LSTM(units=50))
model_solar.add(Dense(1))
model_solar.compile(optimizer='adam', loss='mean_squared_error')

In [ ]:
# Train the model
history_solar = model_solar.fit(X_solar_train, y_solar_train, epochs=10, batch_size=32, validation_split=0.1)



In [ ]:
# Evaluate the model
solar_test_loss = model_solar.evaluate(X_solar_test, y_solar_test)

In [ ]:
# Make predictions
y_solar_pred = model_solar.predict(X_solar_test)
y_solar_pred = solar_scaler.inverse_transform(np.concatenate((np.zeros((y_solar_pred.shape[0], 1)), y_solar_pred), axis=1))[:, 1]

In [ ]:
# Build the RNN model for wind generation
model_wind = Sequential()
model_wind.add(LSTM(units=50, return_sequences=True, input_shape=(X_wind_train.shape[1], X_wind_train.shape[2])))
model_wind.add(LSTM(units=50))
model_wind.add(Dense(1))
model_wind.compile(optimizer='adam', loss='mean_squared_error')

In [ ]:
# Train the model
history_wind = model_wind.fit(X_wind_train, y_wind_train, epochs=5, batch_size=32, validation_split=0.1)


In [ ]:

# Evaluate the model
wind_test_loss = model_wind.evaluate(X_wind_test, y_wind_test)


In [ ]:

# Make predictions
y_wind_pred = model_wind.predict(X_wind_test)
y_wind_pred = wind_scaler.inverse_transform(np.concatenate((np.zeros((y_wind_pred.shape[0], 1)), y_wind_pred), axis=1))[:, 1]


In [ ]:

# Plotting predictions vs actual values for solar
plt.figure(figsize=(14, 7))
plt.plot(y_solar_test, label='Actual Solar Generation')
plt.plot(y_solar_pred, label='Predicted Solar Generation')
plt.title('Solar Generation - Actual vs Predicted')
plt.xlabel('Time')
plt.ylabel('Generation (MW)')
plt.legend()
plt.show()


In [ ]:

# Plotting predictions vs actual values for wind
plt.figure(figsize=(14, 7))
plt.plot(y_wind_test, label='Actual Wind Generation')
plt.plot(y_wind_pred, label='Predicted Wind Generation')
plt.title('Wind Generation - Actual vs Predicted')
plt.xlabel('Time')
plt.ylabel('Generation (MW)')
plt.legend()
plt.show()

In [ ]:

from lime.lime_tabular import LimeTabularExplainer


In [ ]:

# Assuming your data is already preprocessed and split into X_train, X_test, y_train, y_test

# Create a LIME explainer
explainer = LimeTabularExplainer(
    training_data=np.array(X1_train),  # Use the training data
    feature_names=['DE_solar_profile'],  # Name of the feature(s)
    class_names=['DE_solar_generation_actual'],  # Name of the target variable
    mode='regression'
)


In [ ]:

# Pick an instance from the test set to explain
i = 5  # You can pick any index of a sample from the test set
instance = X1_test[i].reshape(1, -1)


In [ ]:

# Explain the instance prediction with Lasso model
exp = explainer.explain_instance(
    data_row=instance.flatten(),
    predict_fn=lasso.predict
)


In [ ]:

# Display the explanation
exp.show_in_notebook(show_table=True)


In [ ]:

# Similarly, you can do it for the wind model
explainer_wind = LimeTabularExplainer(
    training_data=np.array(X2_train),
    feature_names=['DE_wind_profile'],
    class_names=['DE_wind_generation_actual'],
    mode='regression'
)

instance_wind = X2_test[5].reshape(1, -1)
exp_wind = explainer_wind.explain_instance(
    data_row=instance_wind.flatten(),
    predict_fn=lasso.predict
)

exp_wind.show_in_notebook(show_table=True)


In [ ]:
import numpy as np
from lime.lime_tabular import LimeTabularExplainer

# Flatten the training data for LIME
X_solar_train_flat = X_solar_train.reshape(X_solar_train.shape[0], -1)

# Create a LIME explainer with the flattened data
explainer_lstm_solar = LimeTabularExplainer(
    training_data=X_solar_train_flat,
    mode='regression',
    feature_names=[f"t{j}_t-{i}" for i in range(X_solar_train.shape[1]) for j in range(X_solar_train.shape[2])],
    discretize_continuous=True
)

# Pick an instance from the test set
i = 0  # Index of the instance
instance_lstm_solar = X_solar_test[i].reshape(1, -1)  # Flatten the instance

# Explain the instance
exp_lstm_solar = explainer_lstm_solar.explain_instance(
    data_row=instance_lstm_solar.flatten(),  # Flatten data_row
    predict_fn=lambda x: model_solar.predict(x.reshape(-1, X_solar_train.shape[1], X_solar_train.shape[2])).flatten()
)

# Display the explanation
exp_lstm_solar.show_in_notebook(show_table=True)


In [ ]:
import numpy as np
from lime.lime_tabular import LimeTabularExplainer

# Flatten the training data for LIME (Wind)
X_wind_train_flat = X_wind_train.reshape(X_wind_train.shape[0], -1)

# Create a LIME explainer with the flattened data (Wind)
explainer_lstm_wind = LimeTabularExplainer(
    training_data=X_wind_train_flat,
    mode='regression',
    feature_names=[f"t{j}_t-{i}" for i in range(X_wind_train.shape[1]) for j in range(X_wind_train.shape[2])],
    discretize_continuous=True
)

# Pick an instance from the test set (Wind)
i = 0  # Index of the instance
instance_lstm_wind = X_wind_test[i].reshape(1, -1)  # Flatten the instance

# Explain the instance (Wind)
exp_lstm_wind = explainer_lstm_wind.explain_instance(
    data_row=instance_lstm_wind.flatten(),  # Flatten data_row
    predict_fn=lambda x: model_wind.predict(x.reshape(-1, X_wind_train.shape[1], X_wind_train.shape[2])).flatten()
)

# Display the explanation (Wind)
exp_lstm_wind.show_in_notebook(show_table=True)
